# RAG CoT
(Chain of Thought)

RAG 파이프라인에서 LLM이 단순한 정보 조합을 넘어서 단계적 사고를 통해 논리적 답변을 할 수 있도록 한다.

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [4]:
# 가상 검색기
from langchain_core.documents import Document

def retrieve_vectordb(query=None):
    return [
        Document(page_content='대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다.'),
        Document(page_content='서울의 대표적인 관광지는 경복궁, 남산타워, 명동 등이 있습니다.'),
        Document(page_content='서울의 인구는 약 천만 명이고, 교통 문화 인프라가 잘 갖추어져 있습니다.')
    ]

retrieve_vectordb()

[Document(metadata={}, page_content='대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다.'),
 Document(metadata={}, page_content='서울의 대표적인 관광지는 경복궁, 남산타워, 명동 등이 있습니다.'),
 Document(metadata={}, page_content='서울의 인구는 약 천만 명이고, 교통 문화 인프라가 잘 갖추어져 있습니다.')]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser  # 출력 -> 문자열 파싱

llm = init_chat_model('gpt-5.6-luna')
prompt = ChatPromptTemplate.from_template('''
당신은 데이터를 분석해서 논리적인 결론을 도출하는 전문가 챗봇입니다.
아래 [검색된 문서]를 바탕으로 사용자의 [질문]에 대해 답변하세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
다음의 단계에 따라 사고한 후, 답변을 작성하세요.
1. **핵심데이터 정리**: 문서에서 사용자 질문과 관련한 팩트를 추출해보세요.
2. **상호관계 분석**: 각 항목별로 어떤 상관관계/시너지를 도출하는지 고민하세요.
3. **논리적 서술**: 위의 사고한 내용을 토대로 사용자 질문에 대한 답변을 준비하세요.
4. **최종 답변**: 서론-본론-결론 구조에 맞춰 완성된 답변을 작성하세요.
''')

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

question = '서울의 인구, 관광지, 교통인프라를 종합해서 여행하기 좋은 이유를 논리적으로 설명해주세요.'
retrieved_docs = retrieve_vectordb(question)
# 문서 본문만 뽑아서 하나의 문자열 context로 생성
context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

response = chain.invoke({'context': context, 'question': question})
print(response)

### 서론  
서울은 약 천만 명이 거주하는 대도시로, 다양한 관광지와 잘 갖추어진 교통·문화·인프라를 갖추고 있어 여행하기 좋은 도시입니다.

### 본론  
첫째, 서울은 약 천만 명의 인구를 바탕으로 도시 규모가 크고 활력이 넘칩니다. 많은 사람이 생활하는 도시인 만큼 여행객이 이용할 수 있는 문화·생활 인프라가 잘 발달해 있어 편리한 여행 환경을 제공합니다.

둘째, 서울에는 경복궁, 남산타워, 명동 등 서로 다른 매력을 지닌 대표 관광지가 있습니다. 경복궁에서는 역사와 전통문화를 경험할 수 있고, 남산타워에서는 서울의 도시 경관을 감상할 수 있으며, 명동에서는 도심의 활기와 다양한 관광 경험을 즐길 수 있습니다. 이처럼 관광지의 성격이 다양해 여행객이 한 도시 안에서 여러 가지 활동을 할 수 있습니다.

셋째, 서울은 교통 인프라가 잘 갖추어져 있어 여러 관광지를 이동하기에 편리합니다. 경복궁, 남산타워, 명동처럼 서로 다른 장소를 효율적으로 방문할 수 있으므로 여행 시간과 이동에 대한 부담을 줄일 수 있습니다. 또한 잘 발달한 문화·생활 인프라는 여행 중 필요한 편의시설을 이용하는 데에도 도움을 줍니다.

### 결론  
종합하면, 서울은 많은 인구를 기반으로 한 활기찬 도시 환경, 역사·경관·쇼핑 등 다양한 관광지, 그리고 편리한 교통·문화 인프라가 서로 시너지를 이루는 곳입니다. 따라서 한 도시에서 다양한 볼거리와 경험을 편리하게 즐길 수 있다는 점에서 여행하기 좋은 도시라고 할 수 있습니다.
